In [1]:
## cnn pipeline with raw sequence inputs
## one-hot encoded

In [25]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np
from tqdm import tqdm


In [26]:
# Mapping for 20 standard amino acids
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

# Directory setup
os.makedirs("cnn_models", exist_ok=True)
os.makedirs("cnn_results", exist_ok=True)



In [27]:
# CNN model
class ProteinCNN(nn.Module):
    def __init__(self, input_dim=20):
        super(ProteinCNN, self).__init__()
        self.conv1 = nn.Conv1d(input_dim, 64, kernel_size=5, padding=2)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x).squeeze(-1)
        x = self.sigmoid(self.fc(x))
        return x.squeeze()

# Dataset class
class ProteinDataset(Dataset):
    def __init__(self, sequences, labels, max_len):
        self.sequences = sequences
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        onehot = np.zeros((20, self.max_len), dtype=np.float32)
        for i, aa in enumerate(seq[:self.max_len]):
            if aa in AA_TO_INDEX:
                onehot[AA_TO_INDEX[aa], i] = 1.0
        return torch.tensor(onehot), torch.tensor(label, dtype=torch.float32)



In [28]:
# Training loop
def train_eval_model(gene_name, df, max_len):
    df = df[df["Frameshift_Mutation"] == 0]
    df = df[df["Phenotype"].isin(["R", "S"])]
    sequences = df["Protein_Sequence"].tolist()
    labels = (df["Phenotype"] == "R").astype(int).tolist()

    dataset = ProteinDataset(sequences, labels, max_len)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)

    model = ProteinCNN()
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    model.train()
    for epoch in range(10):
        for X, y in loader:
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X, y in loader:
            out = model(X)
            all_preds.extend(out.numpy())
            all_labels.extend(y.numpy())
    auc = roc_auc_score(all_labels, all_preds)

    torch.save(model.state_dict(), f"cnn_models/{gene_name}_cnn.pt")

    result_df = pd.DataFrame({
        "Gene": [gene_name],
        "AUC": [auc]
    })
    result_df.to_csv(f"cnn_results/{gene_name}_results.csv", index=False)
    return result_df



In [29]:
gene_list=['rpoB','rpsL','tlyA','pncA','eis','gid','katG','inhA','embA','embB', 'embC','gyrB', 'gyrA', 'ethA', 'ethR']

In [30]:
all_results = []

for gene in tqdm(gene_list):
    file_path = f"./data/sequence_data_csv/{gene}_combined_sequence_data.csv"
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        max_len = df["Protein_Sequence"].str.len().max()
        result = train_eval_model(gene, df, max_len)
        all_results.append(result)
print(all_results)
summary_df = pd.concat(all_results, ignore_index=True)
summary_df.to_csv("cnn_results/summary_auc_scores.csv", index=False)


100%|██████████| 15/15 [08:30<00:00, 34.03s/it]

[   Gene       AUC
0  rpoB  0.963639,    Gene       AUC
0  rpsL  0.767147,    Gene       AUC
0  tlyA  0.507306,    Gene       AUC
0  pncA  0.849429,   Gene      AUC
0  eis  0.52025,   Gene       AUC
0  gid  0.750087,    Gene       AUC
0  katG  0.894722,    Gene       AUC
0  inhA  0.535999,    Gene       AUC
0  embA  0.546183,    Gene       AUC
0  embB  0.922261,    Gene       AUC
0  embC  0.654951,    Gene      AUC
0  gyrB  0.51721,    Gene       AUC
0  gyrA  0.883233,    Gene       AUC
0  ethA  0.592663,    Gene       AUC
0  ethR  0.512791]


## per residue attribution

In [31]:
# Enable gradient tracking on input.

# Run a forward pass to get model predictions.

# Backpropagate using .backward() on the prediction score (or loss).

# Extract gradients w.r.t. input → these act as importance scores.

# Aggregate across channels (i.e., amino acids) per residue.

# Store per-residue importances across the dataset.

In [34]:
def compute_residue_importance(model, dataloader, device, max_len, save_path=None):
    model.eval()
    model.to(device)
    all_importance_scores = []
    all_labels = []
    
    for inputs, labels in tqdm(dataloader, desc="Computing Residue Importance"):
        inputs = inputs.to(device)
        inputs.requires_grad_()  # enable grad tracking
        outputs = model(inputs)

        grads = []
        for i in range(len(inputs)):
            model.zero_grad()
            outputs[i].backward(retain_graph=True)

            # input gradient: shape (20, max_len)
            grad = inputs.grad[i].detach().cpu().numpy()
            score = np.abs(grad).sum(axis=0)  # sum over AA channels → shape (max_len,)
            grads.append(score)
        all_importance_scores.extend(grads)
        all_labels.extend(labels.numpy())

        # Important! Reset grad tracking
        inputs.requires_grad_(False)
        inputs.grad = None

    importance_matrix = np.array(all_importance_scores)  # (N, max_len)
    labels_array = np.array(all_labels)

    if save_path:
        np.savez(save_path, importance=importance_matrix, labels=labels_array)

    return importance_matrix, labels_array


In [40]:
model = ProteinCNN()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:

for gene in tqdm(gene_list):
    model.load_state_dict(torch.load(f"cnn_models/{gene}_cnn.pt"))
    # Load the CSV
    df = pd.read_csv(f"./data/sequence_data_csv/{gene_name}_combined_sequence_data.csv")

    # Filter valid samples
    df = df[df["Frameshift_Mutation"] == 0]
    df = df[df["Phenotype"].isin(["R", "S"])]
    sequences = df["Protein_Sequence"].tolist()
    labels = df["Phenotype"].map({"S": 0, "R": 1}).tolist()

    # Get max_len per gene
    max_len = max(len(seq) for seq in sequences)

    # Create Dataset + DataLoader
    dataset = ProteinDataset(sequences, labels, max_len)
    dataloader = DataLoader(dataset, batch_size=16, shuffle=False)
    
    importance_scores, labels = compute_residue_importance(
        model, dataloader, device, max_len,
        save_path=f"residue_importance/{gene}_cnn_residue_importance.npz"
    )
    # Compute mean over sequences (axis=0)
    global_importance = importance_scores.mean(axis=0)

    # Save mean score as CSV
    df = pd.DataFrame({
        "Residue_Position": list(range(len(global_importance))),
        "Importance": global_importance
    })
    df.to_csv(f"residue_importance/{gene}_cnn_residue_importance_global.csv", index=False)


    R_mask = (labels == 1)
    S_mask = (labels == 0)

    global_R = importance_scores[R_mask].mean(axis=0)
    global_S = importance_scores[S_mask].mean(axis=0)

    pd.DataFrame({
        "Residue_Position": list(range(len(global_R))),
        "Importance_R": global_R,
        "Importance_S": global_S
    }).to_csv(f"residue_importance/{gene}_cnn_residue_importance_R_vs_S.csv", index=False)


In [46]:
def evaluate_topk(imp_df, true_residues, k):
    topk_positions = set(imp_df.iloc[:k]["Residue_Position"])
    tp = len(topk_positions & true_residues)
    fp = k - tp
    fn = len(true_residues - topk_positions)
    
    precision = tp / k if k > 0 else 0
    recall = tp / len(true_residues) if true_residues else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1, tp


In [48]:
## calculate precision recall
for gene in tqdm(gene_list):
    # Load WHO variant catalog
    catalog_df = pd.read_csv("./data/filtered_variants_output.csv")

    # Filter for current gene and R-conferring mutations
    catalog_df = catalog_df.query("gene == @gene and Present_R > 0").copy()

    # Convert WHO amino acid positions from 1-based to 0-based
    catalog_df["aa_pos_0idx"] = catalog_df["aa_pos"] - 1

    # Set of true positive residue positions
    true_residue_set = set(catalog_df["aa_pos_0idx"])
    print(f"WHO R-variants in {gene}: {len(true_residue_set)}")
    
    # load cnn-based residue importance
    # Load the global importance CSV
    imp_df = pd.read_csv(f"residue_importance/{gene}_cnn_residue_importance_global.csv")

    # Sort residues by importance (descending)
    imp_df = imp_df.sort_values("Importance", ascending=False).reset_index(drop=True)
    
    #evaluate across K
    for k in [10, 20, 30, 50, 100]:
        prec, rec, f1, tp = evaluate_topk(imp_df, true_residue_set, k)
        print(f"[{gene_name}] Top-{k:3d} → Precision: {prec:.2f}, Recall: {rec:.2f}, F1: {f1:.2f}, TP: {tp}")

    top_k_df = imp_df.iloc[:k]
    overlap_df = catalog_df[catalog_df["aa_pos_0idx"].isin(top_k_df["Residue_Position"])]

    # Merge to get the importance value for each overlap
    overlap_annotated = overlap_df.merge(top_k_df, left_on="aa_pos_0idx", right_on="Residue_Position")
    overlap_annotated = overlap_annotated.sort_values("Importance", ascending=False)

    # Save to file
    overlap_annotated.to_csv(f"residue_importance/{gene}_cnn_top{k}_overlap_with_WHO.csv", index=False)




 13%|█▎        | 2/15 [00:00<00:00, 16.18it/s]

WHO R-variants in rpoB: 325
[rpsL] Top- 10 → Precision: 0.00, Recall: 0.00, F1: 0.00, TP: 0
[rpsL] Top- 20 → Precision: 0.05, Recall: 0.00, F1: 0.01, TP: 1
[rpsL] Top- 30 → Precision: 0.10, Recall: 0.01, F1: 0.02, TP: 3
[rpsL] Top- 50 → Precision: 0.22, Recall: 0.03, F1: 0.06, TP: 11
[rpsL] Top-100 → Precision: 0.23, Recall: 0.07, F1: 0.11, TP: 23
WHO R-variants in rpsL: 16
[rpsL] Top- 10 → Precision: 0.40, Recall: 0.25, F1: 0.31, TP: 4
[rpsL] Top- 20 → Precision: 0.30, Recall: 0.38, F1: 0.33, TP: 6
[rpsL] Top- 30 → Precision: 0.27, Recall: 0.50, F1: 0.35, TP: 8
[rpsL] Top- 50 → Precision: 0.20, Recall: 0.62, F1: 0.30, TP: 10
[rpsL] Top-100 → Precision: 0.14, Recall: 0.88, F1: 0.24, TP: 14
WHO R-variants in tlyA: 34
[rpsL] Top- 10 → Precision: 0.20, Recall: 0.06, F1: 0.09, TP: 2
[rpsL] Top- 20 → Precision: 0.15, Recall: 0.09, F1: 0.11, TP: 3
[rpsL] Top- 30 → Precision: 0.13, Recall: 0.12, F1: 0.12, TP: 4
[rpsL] Top- 50 → Precision: 0.20, Recall: 0.29, F1: 0.24, TP: 10
[rpsL] Top-100 → 

 60%|██████    | 9/15 [00:00<00:00, 25.86it/s]

WHO R-variants in gid: 155
[rpsL] Top- 10 → Precision: 1.00, Recall: 0.06, F1: 0.12, TP: 10
[rpsL] Top- 20 → Precision: 0.80, Recall: 0.10, F1: 0.18, TP: 16
[rpsL] Top- 30 → Precision: 0.80, Recall: 0.15, F1: 0.26, TP: 24
[rpsL] Top- 50 → Precision: 0.76, Recall: 0.25, F1: 0.37, TP: 38
[rpsL] Top-100 → Precision: 0.74, Recall: 0.48, F1: 0.58, TP: 74
WHO R-variants in katG: 391
[rpsL] Top- 10 → Precision: 0.60, Recall: 0.02, F1: 0.03, TP: 6
[rpsL] Top- 20 → Precision: 0.55, Recall: 0.03, F1: 0.05, TP: 11
[rpsL] Top- 30 → Precision: 0.47, Recall: 0.04, F1: 0.07, TP: 14
[rpsL] Top- 50 → Precision: 0.44, Recall: 0.06, F1: 0.10, TP: 22
[rpsL] Top-100 → Precision: 0.45, Recall: 0.12, F1: 0.18, TP: 45
WHO R-variants in inhA: 44
[rpsL] Top- 10 → Precision: 0.30, Recall: 0.07, F1: 0.11, TP: 3
[rpsL] Top- 20 → Precision: 0.30, Recall: 0.14, F1: 0.19, TP: 6
[rpsL] Top- 30 → Precision: 0.27, Recall: 0.18, F1: 0.22, TP: 8
[rpsL] Top- 50 → Precision: 0.26, Recall: 0.30, F1: 0.28, TP: 13
[rpsL] Top-1

100%|██████████| 15/15 [00:00<00:00, 25.36it/s]

WHO R-variants in gyrA: 106
[rpsL] Top- 10 → Precision: 0.10, Recall: 0.01, F1: 0.02, TP: 1
[rpsL] Top- 20 → Precision: 0.05, Recall: 0.01, F1: 0.02, TP: 1
[rpsL] Top- 30 → Precision: 0.10, Recall: 0.03, F1: 0.04, TP: 3
[rpsL] Top- 50 → Precision: 0.14, Recall: 0.07, F1: 0.09, TP: 7
[rpsL] Top-100 → Precision: 0.14, Recall: 0.13, F1: 0.14, TP: 14
WHO R-variants in ethA: 221
[rpsL] Top- 10 → Precision: 0.40, Recall: 0.02, F1: 0.03, TP: 4
[rpsL] Top- 20 → Precision: 0.40, Recall: 0.04, F1: 0.07, TP: 8
[rpsL] Top- 30 → Precision: 0.47, Recall: 0.06, F1: 0.11, TP: 14
[rpsL] Top- 50 → Precision: 0.50, Recall: 0.11, F1: 0.18, TP: 25
[rpsL] Top-100 → Precision: 0.54, Recall: 0.24, F1: 0.34, TP: 54
WHO R-variants in ethR: 20
[rpsL] Top- 10 → Precision: 0.30, Recall: 0.15, F1: 0.20, TP: 3
[rpsL] Top- 20 → Precision: 0.20, Recall: 0.20, F1: 0.20, TP: 4
[rpsL] Top- 30 → Precision: 0.23, Recall: 0.35, F1: 0.28, TP: 7
[rpsL] Top- 50 → Precision: 0.18, Recall: 0.45, F1: 0.26, TP: 9
[rpsL] Top-100 → 